# Stemming vs. Lemmatization in NLP
Both are **text normalization** techniques that reduce inflected word forms to a common base, enabling better generalization in NLP models.

| | Stemming | Lemmatization |
|---|---|---|
| **Method** | Remove suffixes by rule | Dictionary lookup + morphology |
| **Output** | May not be a real word | Always a valid dictionary word |
| **Speed** | Faster | Slower (needs model) |
| **Example** | `studies → studi` | `studies → study` |
| **Library** | NLTK (PorterStemmer, SnowballStemmer) | spaCy (`token.lemma_`) |

> **Rule of thumb:** Use stemming for speed (search indexing); use lemmatization for accuracy (chatbots, text classification).

###  Stemming with NLTK — PorterStemmer
The **Porter Stemmer** (1980) applies a sequence of rules to strip common English suffixes. It's the most widely used stemmer for English. Import it from `nltk.stem` and create an instance — no model download needed.

In [4]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

Key observations:
- `eating`, `eats`, `eat` → all stem to `eat` 
- `ate` → remains `ate`  (Porter cannot handle irregular past tense — a known limitation)
- `ability` → `abil`  (not a real word — stemming's trade-off: speed over correctness)

In [6]:
words = ["eating", "eats", "eat", "ate", "adjustable", "rafting", "ability", "meeting"]

for word in words:
    print(word, "|", stemmer.stem(word))

###  Lemmatization in spaCy
spaCy's lemmatizer uses the **attribute_ruler + lemmatizer** pipeline components and looks up the correct lemma based on the word's POS context. Unlike stemming, lemmas are always valid English words.

> Requires `en_core_web_sm` loaded — lemmatization needs the full pipeline (tagger + lemmatizer).

In [9]:
import spacy

Key improvements over stemming:
- `ate → eat`  (correctly handles irregular verb — stemming failed here)
- `ability → ability`  (preserved as a valid word)
- `better → well`  (correctly maps to the base adjective form — stemming cannot do this)

Note: `adjustable → adjustable` (no change when already a base form).

In [13]:
nlp = spacy.load("en_core_web_sm")
doc = nlp("Mando talked for 3 hours although talking isn't his thing")
doc = nlp("eating eats eat ate adjustable rafting ability meeting better")

for token in doc:
    print(token, "|", token.lemma_)

###  Customizing the Lemmatizer — AttributeRuler
The `attribute_ruler` component lets you define custom lemma overrides via `ar.add()`. It accepts a list of token patterns and an attribute dictionary. Use this for domain-specific slang, abbreviations, or brand names the model doesn't know.

The active pipeline: `['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']`. We target `attribute_ruler` because it runs before `lemmatizer` — rules set here override the model's default lemmas.

In [16]:
nlp.pipe_names

`ar.add([[{"TEXT": "Bro"}], [{"TEXT": "Brah"}]], {"LEMMA": "Brother"})` teaches the pipeline that both `Bro` and `Brah` should lemmatize to `Brother`. The pattern is a list of OR-conditions; the attribute dict is `{"LEMMA": target_lemma}`.

In [18]:
ar = nlp.get_pipe('attribute_ruler')

ar.add([[{"TEXT":"Bro"}],[{"TEXT":"Brah"}]],[{"LEMMA":"Brother"}])

doc = nlp("Bro, you wanna go? Brah, don't say no! I am exhausted")
for token in doc:
    print(token.text, "|", token.lemma_)

Verify the custom lemma: `doc[6]` is `"Brah"`, and `doc[6].lemma_` should now return `"Brother"` — confirming the rule override took effect.

In [20]:
doc[6]

In [22]:
doc[6].lemma_